In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from pydantic import BaseModel,Field
from typing import Literal

##Libs for agents
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.tools import tool
from langchain.agents import create_agent
import requests
import os

In [ ]:
##coding,google search,weather

class FlowState(BaseModel):
    question: str = Field(description="User's question")   
    category:Literal["coding","google_search","weather"] = Field(default="google_search")
    answer: str = Field(default="",description="Answer to the question")    

In [ ]:
class QuestionCategory(BaseModel): 
    category:Literal["coding","google_search","weather"] = Field(default="google_search", description ="Question Category")

In [ ]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [ ]:
### Define Your Agents - googleSearchAgent, Weather Agent
search = GoogleSerperAPIWrapper()
tools = [search.run]

google_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a agent and can search for any question on google."
)


## Weather Agent
api_key = os.getenv("OPENWEATHER_API_KEY")


@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    print("weather agent called")

    api_key = os.getenv("OPENWEATHER_API_KEY")

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": city,
        "appid": api_key,
        "units": "metric"
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        return f"Unable to fetch weather for {city}."

    data = response.json()

    return (
        f"Current weather in {data['name']}:\n"
        f"Temperature: {data['main']['temp']}°C\n"
        f"Feels like: {data['main']['feels_like']}°C\n"
        f"Humidity: {data['main']['humidity']}%\n"
        f"Condition: {data['weather'][0]['description']}"
    )   

weather_agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a agent and can provide real time weather details."
)

In [ ]:
print("API Key:", os.getenv("OPENWEATHER_API_KEY"))

In [ ]:
def check_question_category(state:FlowState) -> FlowState:
    st_llm = llm.with_structured_output(QuestionCategory)
    res = st_llm.invoke(f"I want to know the category of my question, question is : {state.question}. If you are not sure then just give 'google_search' as a category")
    state.category = res.category
    return state

In [ ]:
def route(state:FlowState) -> Literal['coding', 'google_search', 'weather']:
    return state.category

In [ ]:
def coding_node(state:FlowState) -> FlowState:
    print("Coding NODE...")
    res = llm.invoke(f"You are a coding expert: {state.question}")
    state.answer = res.content
    return state


def weather_node(state:FlowState) -> FlowState:
    res = weather_agent.invoke({"messages":[
        {"role":"user", "content":state.question}
    ]})
    state.answer = res["messages"][-1].content
    return state


def google_search_node(state:FlowState) -> FlowState:
    res = google_agent.invoke({"messages":[
        {"role":"user", "content":state.question}
    ]})
    state.answer = res["messages"][-1].content
    return state

In [ ]:
graph = StateGraph(FlowState)

graph.add_node("check_question_category", check_question_category) 
graph.add_node("coding", coding_node)
graph.add_node("weather", weather_node) 
graph.add_node("google_search", google_search_node)


graph.add_edge(START, "check_question_category")
graph.add_conditional_edges("check_question_category", route)
graph.add_edge("coding", END)
graph.add_edge("weather", END)  
graph.add_edge("google_search", END)

graph = graph.compile()


In [ ]:
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())

In [ ]:
response=graph.invoke({"question":"What is the weather in yanam?"})

In [ ]:
response